## 18. نرمال‌سازی قیمت

### فروش

برای آگهی فروش با مساحت معتبر:

```text
sale_price_per_sqm = sale_price / valid_building_area
```

قیمت هر مترمربع برای زمین، ویلا، آپارتمان و واحد تجاری ممکن است معنای متفاوتی داشته باشد.
تیم باید مخرج مناسب را برای هر نوع ملک مشخص کند.

### رهن و اجاره

فرض کنید:

- `D`: مبلغ ودیعه
- `R`: اجاره ماهانه
- `k`: اجاره ماهانه معادل هر یک میلیون تومان ودیعه

آنگاه:

```text
equivalent_monthly_rent = R + (D / 1_000_000) * k
equivalent_deposit = D + (R / k) * 1_000_000
```

مقدار `k` ثابت و قطعی نیست. تیم باید:

- مقدار پایه را مستند کند.
- حداقل دو سناریوی جایگزین تعریف کند.
- اثر تغییر `k` بر رتبه‌بندی محله‌ها و نتایج را بررسی کند.
- واحدها را در تمام محاسبات کنترل کند.

In [2]:
import pandas as pd
import numpy as np

<div dir="rtl" align="right">

###  خواندن دیتاست
</div>

In [3]:
df = pd.read_feather("../Outputs/03_df.feather")

<div dir="rtl" align="right">

###  تنظیمات نمایش داده‌ها

</div>

In [4]:
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)
pd.set_option("display.max_colwidth", None)

In [5]:
counts = df.groupby(["cat2_slug", "cat3_slug"]).size()
print(counts)


cat2_slug             cat3_slug                         
commercial-rent       apartment-rent                             0
                      apartment-sell                             0
                      house-villa-rent                           0
                      house-villa-sell                           0
                      industry-agriculture-business-rent      9150
                      industry-agriculture-business-sell         0
                      office-rent                            21412
                      office-sell                                0
                      partnership                                0
                      plot-old                                   0
                      presell                                    0
                      shop-rent                              45970
                      shop-sell                                  0
                      suite-apartment                            0
     

C:\Users\lenovo\AppData\Local\Temp\ipykernel_23832\490707060.py:1: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  counts = df.groupby(["cat2_slug", "cat3_slug"]).size()


<div dir="rtl" align="right">

### تعیین مساحت معتبر

در این بخش تابع `get_area` بر اساس نوع ملک، مساحت مناسب را تعیین می‌کند تا در محاسبات نرمال‌سازی قیمت از مساحت صحیح استفاده شود.

</div>

In [15]:
def get_area(row):
    cat3 = str(row["cat3_slug"])

    if "house-villa" in cat3:
        area = row["land_size"]

        if pd.isna(area) or area <= 0:
            area = row["building_size"]

    else:
        area = row["building_size"]

        if pd.isna(area) or area <= 0:
            area = row["land_size"]

    return area

In [19]:
df["valid_area"] = df.apply(get_area, axis=1)

<div dir="rtl" align="right">

### نرمال‌سازی قیمت

در این بخش تابع `normalize_prices` برای محاسبه شاخص‌های قیمت نرمال‌شده استفاده می‌شود.

برای آگهی‌های فروش، قیمت بر اساس مساحت معتبر به قیمت هر مترمربع تبدیل می‌شود.

برای آگهی‌های رهن و اجاره نیز با استفاده از ضریب تبدیل `k`، مبلغ رهن و اجاره به مقادیر معادل تبدیل می‌شوند تا امکان مقایسه آگهی‌ها در یک مقیاس مشترک فراهم شود.

</div>

In [21]:

def normalize_prices(df, k):

    # =========================
    # Sale
    # =========================

    is_sale = (
        (df["price_regime"] == "sell")
        & (df["price_status"] == "valid")
    )

    # # محاسبه مساحت معتبر
    # df.loc[is_sale, "valid_area"] = (
    #     df.loc[is_sale]
    #     .apply(get_area, axis=1)
    # )

    # فقط مساحت‌های معتبر
    valid_sale_area = (
        is_sale
        & df["valid_area"].notna()
        & (df["valid_area"] > 0)
        & df["sale_price"].notna()
        & (df["sale_price"] > 0)
    )

    # قیمت هر متر مربع
    df.loc[valid_sale_area, "sale_price_per_sqm"] = (
        df.loc[valid_sale_area, "sale_price"]
        / df.loc[valid_sale_area, "valid_area"]
    )


    # =========================
    # Mortgage + Rent
    # =========================

    is_rent = (
        (df["price_regime"] == "mortgage_and_rent")
        & (df["price_status"] == "valid")
    )

    if k > 0:

        df.loc[is_rent, "equivalent_monthly_rent"] = (
            df.loc[is_rent, "monthly_rent"]
            + (
                df.loc[is_rent, "deposit_amount"] / 1_000_000
            ) * k
        )

        df.loc[is_rent, "equivalent_deposit"] = (
            df.loc[is_rent, "deposit_amount"]
            + (
                df.loc[is_rent, "monthly_rent"] / k
            ) * 1_000_000
        )


    # =========================
    # Rent only
    # =========================

    is_rent_only = (
        (df["price_regime"] == "rent_only")
        & (df["price_status"] == "valid")
    )

    df.loc[is_rent_only, "equivalent_monthly_rent"] = (
        df.loc[is_rent_only, "monthly_rent"]
    )


    # =========================
    # Mortgage only
    # =========================

    is_mortgage_only = (
        (df["price_regime"] == "mortgage_only")
        & (df["price_status"] == "valid")
    )

    df.loc[is_mortgage_only, "equivalent_deposit"] = (
        df.loc[is_mortgage_only, "deposit_amount"]
    )


    return df

<div dir="rtl" align="right">

## اجرای نرمال‌سازی قیمت

در این مرحله تابع نرمال‌سازی با مقدار `k = 30,000` اجرا می‌شود و ستون‌های محاسبه‌شده به DataFrame اضافه می‌شوند.

</div>

In [22]:
df = normalize_prices(df, k=30_000)

C:\Users\lenovo\AppData\Local\Temp\ipykernel_23832\2582083780.py:28: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[141666666.66666666 50000000.0 87000000.0 ... 41428571.428571425
 41500000.0 16000000.0]' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.loc[valid_sale_area, "sale_price_per_sqm"] = (


In [23]:
df[
    df["valid_area"].notna()
][
    [
        "cat3_slug",
        "land_size",
        "building_size",
        "valid_area",
        "sale_price",
        "sale_price_per_sqm"
    ]
].head(20)

,cat3_slug,land_size,building_size,valid_area,sale_price,sale_price_per_sqm
0,villa,<NA>,500.0,500.0,NaN,NaN
1,apartment-sell,<NA>,60.0,60.0,8.500000e+09,141666666.666667
2,apartment-rent,<NA>,132.0,132.0,NaN,NaN
3,office-rent,<NA>,90.0,90.0,NaN,NaN
4,apartment-sell,<NA>,115.0,115.0,5.750000e+09,50000000.0
5,apartment-rent,<NA>,100.0,100.0,NaN,NaN
6,office-rent,<NA>,80.0,80.0,NaN,NaN
7,apartment-sell,<NA>,100.0,100.0,8.700000e+09,87000000.0
8,apartment-sell,<NA>,78.0,78.0,6.500000e+08,8333333.333333
9,apartment-sell,<NA>,80.0,80.0,3.000000e+09,37500000.0


In [ ]:
df.to_feather("../Outputs/18_df.feather")